# 1) Import Libraries

In [21]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 2) Load Data

In [22]:
X_train = pd.read_csv(r"../data/processed_data/X_train.csv") 
X_test = pd.read_csv(r"../data/processed_data/X_test.csv")
y_train = pd.read_csv(r"../data/processed_data/y_train.csv")
y_test = pd.read_csv(r"../data/processed_data/y_test.csv")

In [23]:
y_train = y_train.squeeze()
y_test = y_test.squeeze()

# 3) Training Models

We will Training Linear Regression, Random Forest, Gradiant Boosting and SVR and make a comparation for what is the best one

## 3.1) Making a general models

In [24]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR()
}

In [25]:
results = []

for name, model in models.items():

    # Treina
    model.fit(X_train, y_train)

    # Prediz
    y_pred = model.predict(X_test)

    # Calcula métricas
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

In [26]:
results_df = pd.DataFrame(results)

results_df.sort_values("R2", ascending=False)

,Model,MAE,RMSE,R2
1,Random Forest,1.831267,2.612944,0.895242
2,Gradient Boosting,3.294424,4.176587,0.732347
3,SVR,4.768057,6.127462,0.423910
0,Linear Regression,6.431252,7.585691,0.117084


## 3.2) Aprofundating in Random Forest

Perform cross-validation to verify whether the result is the best in this case

In [27]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    RandomForestRegressor(random_state=42),
    X_train,
    y_train,
    cv=5,
    scoring="r2"
)

print(scores)
print(scores.mean())
print(scores.std())

[0.89509378 0.89345534 0.88406962 0.89574331 0.89090175]
0.8918527605637209
0.0042353401396567625


With this, we are certain that this model is the best to use in this situation, with 89% of the target.

## 3.3) Adjusting hyperparameters

In [49]:
import numpy as np

from sklearn.model_selection import RandomizedSearchCV


def tune_model(model, params, X_train, y_train, X_test, y_test,
               n_iter=20, cv=5):

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=n_iter,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        random_state=42
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    y_pred = best_model.predict(X_test)

    print("=" * 50)
    print("Best Parameters:")
    print(search.best_params_)
    print()

    print(f"Best CV R²: {search.best_score_:.4f}")
    print()

    print("Test Metrics")
    print(f"MAE : {mean_absolute_error(y_test, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    print(f"R²  : {r2_score(y_test, y_pred):.4f}")

    return best_model, search.best_params_

In [50]:
params = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

best_rf, best_params = tune_model(
    RandomForestRegressor(random_state=42),
    params,
    X_train,
    y_train,
    X_test,
    y_test
)

Best Parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 30}

Best CV R²: 0.8929

Test Metrics
MAE : 1.8328
RMSE: 2.6100
R²  : 0.8955


With this test, we have certain that this is the best parameters to use in our model with this data.

## 3.4) Verifying overfitting

In [52]:
y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)

print("Train R²:", r2_score(y_train, y_train_pred))
print("Test R²:", r2_score(y_test, y_test_pred))

Train R²: 0.9792627652172504
Test R²: 0.8954800512399949


The model showed a higher R² on the training set (0.9793) than on the test set (0.8955), indicating some degree of overfitting. However, the relatively high test R² demonstrates that the model generalizes well to unseen data.

# 4) Saving model and parameters

In [53]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    best_rf,
    "../models/random_forest_model.pkl"
)

['../models/random_forest_model.pkl']

In [57]:
import json

best_params_json = {
    key: value.item() if hasattr(value, "item") else value
    for key, value in best_params.items()
}

with open("../models/random_forest_params.json", "w") as file:
    json.dump(
        best_params_json,
        file,
        indent=4
    )

# 5) Conclusion

We tested four different regression models: Random Forest, Support Vector Regression (SVR), Linear Regression, and Gradient Boosting. Among them, Random Forest achieved the best performance, with an R² score of approximately 89.5%, and was selected as the best model for this stage of the project.

We then performed hyperparameter optimization to determine whether a better configuration could improve the model's performance. The best configuration found was:

n_estimators = 300
min_samples_split = 5
min_samples_leaf = 1
max_depth = 30

After optimization, the model achieved an R² score of 0.8955, an MAE of 1.8328, and an RMSE of 2.6100.

This model is designed to predict the patient's motor_updrs score based on clinical and voice-related features. The prediction can potentially help provide an estimate of the patient's motor symptom severity and serve as a supporting tool for medical analysis. However, the model does not directly diagnose Parkinson's disease.

The next step is to develop a neural network using PyTorch and compare its performance with the optimized Random Forest model. This comparison will help determine whether a deep learning approach can outperform the classical machine learning model for this regression problem.